# GKP Gate Optimization

Vacuum $\to$ finite-energy $\ket{0_L}$ with three ideal-gate ansätze from `gate_optimization`: SNAP + displacements, ECD + qubit rotations, and conditional squeezing + qubit rotations. Each gate set is swept over the same depth grid, and every run is saved with `gate_results` so a re-executed notebook loads finished depths instead of re-optimizing.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import jax

jax.config.update("jax_enable_x64", True)  # before any array is created

import time
from pathlib import Path

import jax.numpy as jnp
import jaxquantum as jqt
import matplotlib.pyplot as plt
import numpy as np

from gkp_optimal_control.gate_optimization import (
    GateBounds,
    OptimizerConfig,
    TrajectoryPenalty,
    make_displacement,
    optimize_gate_sequence,
)
from gkp_optimal_control.gate_results import (
    build_sequence,
    gate_result_path,
    load_gate_result,
    save_gate_result,
)
from gkp_optimal_control.plotting import (
    plot_photon_number,
    plot_wigner,
    set_plot_style,
)
from gkp_optimal_control.states import gkp_states

set_plot_style()

## Output directory

Sweep results go to `DATA_DIR/sweep/<gate_set>_g<depth>.json` and refined results to `DATA_DIR/refined/`. Filenames carry no fidelity, so a re-run finds and loads them. Set `FORCE_RERUN = True` to re-optimize and overwrite.

In [ ]:
DATA_DIR = Path("../../data/gkp_optimization/vac_to_gkp")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = "vac_to_gkp_optimization.ipynb"
FORCE_RERUN = False

## Settings

In [ ]:
N_FOCK = 200          # truncation; must hold the target plus mid-circuit excursions
N_LEAK = 10           # top Fock levels carrying the leakage penalty
N_SNAP = 10           # SNAP phases per gate (None = all N_FOCK)
DISP_METHOD = "quadrature"   # ECD / SNAP displacements; ~4x faster than "expm"
SQ_METHOD = "eig"            # CSQ squeezers
QUBIT_TARGET = "ground"      # ECD / CSQ: qubit must return to |g>
LOSS_TYPE = "log_infidelity"

TARGET_F = 0.999       # F*

ANSATZ = {
    "snap": dict(disp_method=DISP_METHOD, n_snap=N_SNAP, n_leak=N_LEAK),
    "ecd": dict(disp_method=DISP_METHOD, qubit_target=QUBIT_TARGET, n_leak=N_LEAK),
    "csq": dict(sq_method=SQ_METHOD, qubit_target=QUBIT_TARGET, n_leak=N_LEAK),
}

bounds = GateBounds(
    max_disp=4.0,      # |alpha| (SNAP), |beta| (ECD)
    max_squeeze=0.8,   # |r| (CSQ)
    leakage_weight=5.0,
)

optimizer = OptimizerConfig(
    n_seeds=32,
    n_adam_iters=2000,
    peak_lr=0.03,
    polish=False,
    seed=0,
)

# Long, low-LR single-start run plus L-BFGS-B, for the refinement stage.
optimizer_polish = OptimizerConfig(
    n_seeds=1,
    n_adam_iters=4000,
    peak_lr=0.005,
    polish=True,
    polish_maxiter=2000,
    seed=0,
)

## States

In [ ]:
# Square-lattice GKP.
gkp_alpha = jnp.sqrt(jnp.pi / 2)        # primitive displacement, logical Z axis
gkp_beta = 1j * jnp.sqrt(jnp.pi / 2)    # primitive displacement, logical X axis
gkp_cutoff = 10                         # lattice-sum truncation, (2c+1)^2 peaks
gkp_delta = 0.3
gkp_0, gkp_1 = gkp_states(N_FOCK, gkp_alpha, gkp_beta, gkp_delta, gkp_cutoff)

vac = jqt.basis(N_FOCK, 0)

p_target = np.abs(np.asarray(gkp_0.data).ravel()) ** 2
print(f"Delta = {gkp_delta}  ({-10 * np.log10(gkp_delta**2):.2f} dB),  <n> = {np.arange(N_FOCK) @ p_target:.2f}")
print(f"target weight in top {N_LEAK} Fock levels : {p_target[-N_LEAK:].sum():.1e}")
print(f"target odd-parity weight              : {p_target[1::2].sum():.1e}  (CSQ reaches even parity only)")

## Depths

`n_gates` counts SNAPs, ECDs or conditional squeezers (each layer also carries a displacement or qubit rotation). Parameter counts per depth differ widely: $N \cdot n_\text{snap} + 2(N+1)$ for SNAP, $4N + 2$ for ECD, $5N + 2$ for CSQ.

In [ ]:
depths = [6, 12, 24, 72]

GATE_SETS = ("snap", "ecd", "csq")
STYLE = {
    "snap": dict(label="SNAP + D", marker="o", color="C0"),
    "ecd": dict(label="ECD + R", marker="s", color="C1"),
    "csq": dict(label="CSq + R", marker="^", color="C2"),
}
STRENGTH_LABEL = {"snap": "|alpha|", "ecd": "|beta|", "csq": "|r|"}


def build_seq(gate_set, depth):
    return build_sequence(gate_set, depth, N_FOCK, **ANSATZ[gate_set])


for gs in GATE_SETS:
    print(f"{gs:>4}: " + "  ".join(f"N={d}->{build_seq(gs, d).n_params}p" for d in depths))

## Sweep driver

`run_sweep` runs every depth in `depths` for one gate set, each from `optimizer.n_seeds` independent random initializations (`seq.init`); no depth is seeded from another. `run_gate_opt` checks `DATA_DIR` first and loads a cached result if its truncation and parameter count match the current settings; otherwise it optimizes and saves with `save_gate_result`, recording the ansatz, initial state, and wall time.

In [ ]:
RESULTS: dict[str, dict[int, object]] = {}
TIMINGS: dict[str, dict[int, float]] = {}


def run_gate_opt(gate_set, depth, *, optimizer, bounds=bounds, params0=None,
                 subdir="sweep", force=None, **meta):
    # One optimization (or cache hit). Returns (result, wall_time_s, cached).
    force = FORCE_RERUN if force is None else force
    path = gate_result_path(DATA_DIR / subdir, gate_set, depth)
    seq = build_seq(gate_set, depth)
    if path.exists() and not force:
        res = load_gate_result(path)
        if res.n_fock == N_FOCK and res.flat_params.size == seq.n_params:
            return res, float(res.metadata.get("wall_time_s", np.nan)), True
        print(f"  cached {path.name} does not match current settings; re-optimizing")

    t0 = time.perf_counter()
    res = optimize_gate_sequence(
        seq, vac, gkp_0,
        loss_type=LOSS_TYPE,
        bounds=bounds,
        optimizer=optimizer,
        params0=params0,
        verbose=False,
    )
    elapsed = time.perf_counter() - t0
    save_gate_result(
        res, path,
        initial_state=vac,
        ansatz=ANSATZ[gate_set],
        overwrite=True,
        source_notebook=SOURCE,
        target="gkp_0",
        gkp_delta=float(gkp_delta),
        wall_time_s=elapsed,
        warm_started=params0 is not None,
        **meta,
    )
    return res, elapsed, False


def run_sweep(gate_set, depths=depths, *, optimizer=optimizer, bounds=bounds):
    # Random-init sweep over depths for one gate set; fills RESULTS[gate_set] / TIMINGS[gate_set].
    store = RESULTS.setdefault(gate_set, {})
    times = TIMINGS.setdefault(gate_set, {})
    print(f"=== {gate_set} | depths {list(depths)} | target F = {TARGET_F} ===")
    for depth in sorted(depths):
        res, elapsed, cached = run_gate_opt(gate_set, depth, optimizer=optimizer, bounds=bounds)
        store[depth], times[depth] = res, elapsed
        print(
            f"  N = {depth:>2}  params = {res.flat_params.size:>5}  "
            f"1 - F = {1 - res.fidelity:.3e}  leak = {res.leakage:.1e}  "
            f"peak {STRENGTH_LABEL[gate_set]} = {res.params['strength_magnitudes'].max():.2f}  "
            f"({'cached' if cached else f'{elapsed:.0f}s'})"
        )
    print()
    return store


def curve(gate_set):
    # Sorted (depths, infidelities, n_params) for one sweep.
    store = RESULTS.get(gate_set, {})
    ds = np.array(sorted(store))
    infid = np.array([1 - store[d].fidelity for d in ds])
    n_par = np.array([store[d].flat_params.size for d in ds])
    return ds, infid, n_par


def min_depth(gate_set, target_f=TARGET_F):
    # Smallest swept depth clearing target_f, or None.
    ds, infid, _ = curve(gate_set)
    ok = ds[infid <= 1 - target_f]
    return int(ok[0]) if ok.size else None


def best_result(gate_set, target_f=TARGET_F):
    # (depth, result) at min_depth if the threshold was met, else the highest-F run.
    store = RESULTS[gate_set]
    d = min_depth(gate_set, target_f)
    if d is None:
        d = max(store, key=lambda k: store[k].fidelity)
    return d, store[d]

## Loop 1 — SNAP + displacements

$$U = D(\alpha_{N+1}) \prod_{i=N}^{1} S(\vec\theta_i)\, D(\alpha_i), \qquad S(\vec\theta) = \sum_n e^{i\theta_n}\ket{n}\bra{n}.$$

In [ ]:
run_sweep("snap")

## Loop 2 — ECD + qubit rotations

$$U = R(\theta_{N+1},\phi_{N+1}) \prod_{i=N}^{1} \mathrm{ECD}(\beta_i)\, R(\theta_i,\phi_i),
\qquad \mathrm{ECD}(\beta) = D(\beta/2)\ket{e}\bra{g} + D(-\beta/2)\ket{g}\bra{e}.$$

In [ ]:
run_sweep("ecd")

## Loop 3 — Conditional squeezing + qubit rotations

$$U = R(\theta_{N+1},\phi_{N+1}) \prod_{i=N}^{1} \mathrm{CSq}(r_i, \varphi_{0,i}, \varphi_{1,i})\, R(\theta_i,\phi_i),
\qquad \mathrm{CSq} = \ket{g}\bra{g}\otimes S(r,\varphi_0) + \ket{e}\bra{e}\otimes S(r,\varphi_1).$$

In [ ]:
run_sweep("csq")

## Resource table

Minimum depth clearing each threshold, and what the $F_\star$ circuit costs. `peak` is the largest gate strength ($|\alpha|$, $|\beta|$ or $|r|$); for ECD and CSQ it sets the gate duration and hence the ancilla-decoherence exposure. `q purity` is the final qubit purity (1 means fully disentangled).

In [ ]:
header = (
    f"{'set':<5} {'d(F*)':>6} {'d(F**)':>7} {'params':>7} {'1-F':>10} "
    f"{'peak':>6} {'leak':>9} {'q purity':>9} {'seed spread':>18} {'time/run':>9}"
)
print(header)
print("-" * len(header))
for gate_set in GATE_SETS:
    if gate_set not in RESULTS:
        continue
    d, res = best_result(gate_set)
    met = min_depth(gate_set) is not None
    d_stretch = min_depth(gate_set, STRETCH_F)
    spread = f"{res.per_seed_fidelity.min():.4f}..{res.per_seed_fidelity.max():.4f}"
    purity = "-" if res.qubit_purity is None else f"{res.qubit_purity:.4f}"
    print(
        f"{gate_set:<5} "
        f"{str(d) + ('' if met else '*'):>6} "
        f"{('-' if d_stretch is None else str(d_stretch)):>7} "
        f"{res.flat_params.size:>7} {1 - res.fidelity:>10.2e} "
        f"{res.params['strength_magnitudes'].max():>6.2f} {res.leakage:>9.1e} "
        f"{purity:>9} {spread:>18} {TIMINGS[gate_set][d]:>8.0f}s"
    )
print(f"\nF* = {TARGET_F}, F** = {STRETCH_F}. Depths marked * did not reach F* on the "
      "swept grid; the row shows the best depth tried.")

## Refining the winning circuits

Each minimum-depth solution is warm-started once with a long, low-learning-rate run plus an L-BFGS-B polish (`optimizer_polish`). This separates *ansatz* capacity from *optimizer* convergence: if a depth that missed threshold clears it after refinement, the sweep budget was the binding constraint and the depth curves above are pessimistic. Refined results are saved under `DATA_DIR/refined/`.

In [ ]:
REFINED: dict[str, object] = {}

for gate_set in GATE_SETS:
    if gate_set not in RESULTS:
        continue
    d, res = best_result(gate_set)
    ref, elapsed, cached = run_gate_opt(
        gate_set, d,
        optimizer=optimizer_polish,
        params0=np.asarray(res.flat_params),
        subdir="refined",
    )
    keep = ref if ref.fidelity >= res.fidelity else res
    REFINED[gate_set] = keep
    print(
        f"{gate_set:>4} | N = {d:>2} : 1 - F  {1 - res.fidelity:.3e} -> {1 - ref.fidelity:.3e}"
        + ("  (cached)" if cached else f"  ({elapsed:.0f}s)")
        + ("" if keep is ref else "  (refinement rejected)")
    )